# Colab GPU Trong IDE (Git Sync): Train LoRA Minh Khang + Giọng Biểu Cảm + Video Hành Động Wan 2.2

Đã tối ưu chạy mượt trên cả **Tesla T4 (15GB)** lẫn **NVIDIA L4 (24GB)**, loại bỏ lỗi build `scipy`/`oyaml` của `ai-toolkit`:
1. **Cell 1:** Pull code & 12 ảnh mới nhất từ GitHub (`nguyenkhanhan123/Test`) + kiểm tra GPU.
2. **Cell 2:** Cài `diffusers + peft + bitsandbytes` đồng bộ và train SDXL LoRA (`pytorch_lora_weights.safetensors`, ~5–8 phút).
3. **Cell 3:** Dùng LoRA vừa train sinh Keyframe hành động mới cho `shot_103`, `shot_104`.
4. **Cell 4:** Audio-First — tạo giọng biểu cảm & khóa thời lượng từng shot.
5. **Cell 5:** Render cảnh hành động thật bằng `Wan-AI/Wan2.2-TI2V-5B-Diffusers` (tự chỉnh FP16 + sequential offload trên T4).
6. **Cell 6:** Ghép video hoàn chỉnh, phát xem ngay trong IDE & tự động ngắt GPU.

In [ ]:
# CELL 1: ĐỒNG BỘ GIT + KIỂM TRA GPU + CÀI THƯ VIỆN CỐT LÕI
import os
REPO_URL = 'https://github.com/nguyenkhanhan123/Test.git'
REPO_DIR = '/content/Test'
PROJECT_DIR = '/content/Test/ai_film_pipeline'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin main && git reset --hard origin/main

%cd {PROJECT_DIR}
!nvidia-smi
!pip install -q --upgrade edge-tts pysubs2 pydantic imageio-ffmpeg mutagen peft bitsandbytes accelerate transformers git+https://github.com/huggingface/diffusers.git
!python prepare_user_assets.py --trigger nv_minhkhang

In [ ]:
# CELL 2: TRAIN CHARACTER LORA TỪ 12 ẢNH (DIFFUSERS + PEFT + ADAMW8BIT — TƯƠNG THÍCH T4 & L4)
%cd /content/Test/ai_film_pipeline
!python cloud_scripts/train_sdxl_lora_peft.py \
  --dataset user_inputs/photos \
  --output loras/nv_minhkhang_lora_v1 \
  --steps 360 \
  --resolution 768

In [ ]:
# CELL 3: DÙNG LORA VỪA TRAIN SINH KEYFRAME HÀNH ĐỘNG MỚI CHO SHOT_103 & SHOT_104
%cd /content/Test/ai_film_pipeline
import gc, glob, json, torch
from pathlib import Path
from diffusers import AutoencoderKL, StableDiffusionXLPipeline

lora_files = sorted(glob.glob('loras/nv_minhkhang_lora_v1/*.safetensors'))
assert lora_files, 'Chưa thấy file LoRA .safetensors sau Cell 2!'

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=dtype)
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', vae=vae, torch_dtype=dtype
).to('cuda')
pipe.load_lora_weights('loras/nv_minhkhang_lora_v1', weight_name=Path(lora_files[-1]).name)

prompts = {
    'shot_103': 'nv_minhkhang, dynamic martial arts forward dash slash, glowing crimson energy arc, rain sparks, anime film still, wide 16:9',
    'shot_104': 'nv_minhkhang, high-speed mid-air counter strike against glowing red portal energy, dramatic rim lighting, anime film still, wide 16:9',
}
out_dir = Path('assets/keyframes_lora')
out_dir.mkdir(parents=True, exist_ok=True)
sc_path = Path('script_action_demo.json')
sc = json.loads(sc_path.read_text(encoding='utf-8'))
for shot in sc['shots']:
    sid = shot['shot_id']
    if sid in prompts:
        img = pipe(prompts[sid], width=1024, height=576, num_inference_steps=28, guidance_scale=7.0).images[0]
        img.save(out_dir / f'{sid}.png')
        shot['keyframe_path'] = f'assets/keyframes_lora/{sid}.png'
        print('Saved LoRA keyframe:', out_dir / f'{sid}.png')
sc_path.write_text(json.dumps(sc, indent=2, ensure_ascii=False), encoding='utf-8')
del pipe, vae
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# CELL 4: AUDIO-FIRST — TẠO GIỌNG BIỂU CẢM & KHÓA THỜI LƯỢNG TỪNG SHOT
%cd /content/Test/ai_film_pipeline
from pathlib import Path
voices_ready = list(Path('user_inputs/voices').glob('*.wav'))
if voices_ready:
    !pip install -q git+https://github.com/danhtran2mind/F5-TTS.git whisperx
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='danhtran2mind/Vi-F5-TTS', local_dir='/content/Vi-F5-TTS')
    from cloud_scripts.step3_cloud_vi_f5_tts import clone_episode_voices_vi_f5
    clone_episode_voices_vi_f5(Path('.'), Path('script_action_demo.json'), Path('/content/Vi-F5-TTS'))
else:
    print('Dùng Edge TTS biểu cảm đa sắc thái (thì thầm / căng thẳng / hô chiến đấu / lắng dịu)...')
    !python step_action_demo.py --tts-only

In [ ]:
# CELL 5: RENDER VIDEO HÀNH ĐỘNG THẬT BẰNG WAN 2.2 TI2V-5B
%cd /content/Test/ai_film_pipeline
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python cloud_scripts/step5_batch_wan22_diffusers.py --script script_action_demo.json --steps 24

In [ ]:
# CELL 6: GHÉP VIDEO HOÀN CHỈNH, PHÁT XEM NGAY TRONG IDE & TỰ ĐỘNG NGẮT GPU
%cd /content/Test/ai_film_pipeline
from IPython.display import Video, display

try:
    !python step_action_demo.py --skip-tts
    display(Video('output/action_pilot_demo.mp4', embed=True, width=960))
finally:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass